# 03 Minimum-Cost Timeline Repair

Author: Endri Sojeva

Find a consistent alternative by selecting an ordered, single-use sequence from an explicit edit catalog. Every intermediate graph is evaluated under the same consistency rules.

## Setup

Use the project's Python environment. Kaggle loads the package and scenario files from a private source snapshot tied to a verified GitHub commit.

In [ ]:
import platform
import subprocess
from importlib.metadata import version
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display
from timetravel.detection import detect_paradoxes
from timetravel.repair import RepairStatus, SearchLimits, repair_timeline
from timetravel.scenarios import load_scenario
from timetravel.visualization import draw_timeline

source_commit = globals().get("SOURCE_COMMIT")
if source_commit is None:
    try:
        source_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        source_commit = "local checkout (commit unavailable)"
print({"source_commit": source_commit, "python": platform.python_version(),
       **{name: version(name) for name in ("networkx", "matplotlib", "pandas")}})
examples = globals().get("SCENARIO_DIR")
if examples is None:
    examples = next(directory / "data/examples" for directory in (Path.cwd(), Path.cwd().parent)
                    if (directory / "data/examples").is_dir())

## Cost model and search

Each edit is charged its nonnegative fee at the current graph. This demonstration uses the default catalog of event and dependency removals. Protected events cannot be removed, modified, or rescheduled. Candidate and time limits bound search effort, while the alternative limit bounds returned results.

Greedy prioritizes a strict reduction in violations per unit cost and can take a locally attractive edit. Branch-and-Bound explores ordered edit prefixes. Their cumulative nonnegative costs are lower bounds for continuations, and an incumbent can prune prefixes exceeding the requested alternatives' cost threshold.

In [ ]:
limits = SearchLimits(max_candidates=5000, max_seconds=10, max_alternatives=1)
rows, scenarios, results = [], {}, {}
for name in ("repair_tradeoff", "grandfather_paradox", "contradictory_timeline", "consistent_loop"):
    scenario = scenarios[name] = load_scenario(examples / f"{name}.json")
    for method in ("greedy", "branch_and_bound"):
        result = results[name, method] = repair_timeline(
            scenario.graph, scenario.constraints, method=method, limits=limits,
        )
        assert result.best and detect_paradoxes(result.best.graph, scenario.constraints).consistent
        rows.append((name, method, str(result.best.total_cost), result.optimality_proven,
                     result.metrics.expanded, result.metrics.pruned, result.metrics.candidates_explored))

summary = pd.DataFrame(rows, columns=["scenario", "method", "cost", "proven", "expanded", "pruned", "candidates"])
display(summary)
print(summary.to_string(index=False))
assert results["repair_tradeoff", "greedy"].best.total_cost == 6
assert results["repair_tradeoff", "branch_and_bound"].best.total_cost == 5
assert results["repair_tradeoff", "branch_and_bound"].optimality_proven
assert results["consistent_loop", "branch_and_bound"].best.total_cost == 0

## Infeasible and interrupted runs

Protect both events in the acyclic beacon conflict to obtain an infeasible catalog. Then allow all edits but permit only one attempt, which leaves a feasible incumbent without a proof.

In [ ]:
scenario = scenarios["contradictory_timeline"]
no_solution = repair_timeline(scenario.graph, scenario.constraints, protected_events=list(scenario.graph))
limited = repair_timeline(scenario.graph, scenario.constraints, limits=SearchLimits(1, 10, 1))
assert no_solution.status == RepairStatus.NO_SOLUTION and no_solution.best is None
assert limited.status == RepairStatus.LIMIT_REACHED and limited.best.report.consistent
assert not limited.optimality_proven
print("Protected catalog:", no_solution.status.value)
print("Limited search:", limited.status.value, limited.limit_reason, "incumbent cost", limited.best.total_cost)

## Compare repair costs and inspect the repaired graph

The bars show costs returned by the algorithms for the trade-off fixture. The second figure shows the three reports that remain in its minimum-cost alternative.

In [ ]:
case = "repair_tradeoff"
methods = ("greedy", "branch_and_bound")
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Greedy", "Branch-and-Bound"], [float(results[case, m].best.total_cost) for m in methods],
       color=["#8b4eb5", "#4b88bb"])
ax.set_ylabel("Total repair cost")
ax.set_title("Repair trade-off scenario")

optimal = results[case, "branch_and_bound"].best
repaired = draw_timeline(optimal.graph)
repaired.axes[0].set_title("Minimum-cost alternative")
for figure in (fig, repaired):
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png", bbox_inches="tight")
        display(Image(data=buffer.getvalue()))
    plt.close(figure)
print("Changed events:", optimal.changed_events)

## Interpretation

The exact search saved one cost unit in the trade-off case. The already consistent cycle required no edits. The protected case proved infeasible under the default catalog, while the interrupted case retained a consistent graph without an optimality claim. The table reports candidate attempts, expansions and pruning for these fixtures. A broader performance study belongs to the next phase.